In [ ]:
!pip install -q x-transformers
!pip install -q flash-attn --no-build-isolation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import math
import os
import sys
import subprocess
import hashlib
import gc
from datetime import datetime
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from transformers import RobertaTokenizerFast, get_cosine_schedule_with_warmup, DataCollatorForLanguageModeling
from datasets import load_dataset
from x_transformers import Encoder

# ==========================================
# 1. CONFIGURATION
# ==========================================
# YOUR REPO ID (Created in previous step)
HF_ID = "prism-lab/wikitext-103-prism-32k-seq4k"

# Hyperparameters
VOCAB_SIZE = 32768
SEQ_LEN = 4096
BATCH_SIZE = 8
EPOCHS = 40
LR = 1e-3
D_MODEL = 512
D_BRANCH = 256
DEPTH = 6
RESUME_PATH = None #"/content/drive/MyDrive/PRISM_Experiments/PILLARS_SplitStream_8Layer_20260116_025321_8438ce62/last.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("high")

# ==========================================
# 2. DATA PIPELINE (The "Pro" Way)
# ==========================================
def prepare_data_from_hub():
    print(f"⬇️ Pulling Pre-Tokenized Data from {HF_ID}...")

    # 1. Load Tokenizer (Instant)
    # This pulls the exact tokenizer you uploaded
    tokenizer = RobertaTokenizerFast.from_pretrained(HF_ID)

    # 2. Load Dataset (Instant)
    # This pulls the already chunked/tokenized data
    dataset = load_dataset(HF_ID)

    print(f"✅ Loaded {len(dataset['train'])} training chunks.")

    # 3. Collator
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=0.15
    )

    return dataset, data_collator
# ==========================================
# 3. PRISM ARCHITECTURE (Complex-Valued)
# ==========================================

class ComplexDropout(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        self.p = p
    def forward(self, z):
        if not self.training or self.p == 0.0: return z
        mask = torch.ones_like(z.real)
        mask = F.dropout(mask, self.p, self.training, inplace=False)
        return z * mask

class RobustPhaseNorm(nn.Module):
    def __init__(self, d_model, eps=1e-5):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(d_model))
        self.eps = eps
    def forward(self, x):
        mag = torch.abs(x)
        rms = torch.sqrt(torch.mean(mag**2, dim=-1, keepdim=True) + self.eps)
        return (x / rms) * self.scale

class ModReLU(nn.Module):
    def __init__(self, features):
        super().__init__()
        self.b = nn.Parameter(torch.zeros(features))

    def forward(self, z):
        # 1. FORCE FLOAT32 FOR GEOMETRY
        # We must calculate magnitude in high precision to prevent
        # square-law overflow (Re^2 + Im^2) from killing the gradients.
        z_32 = z.to(torch.complex64)

        # 2. Calculate Magnitude (Safe)
        mag = torch.abs(z_32)

        # 3. Activation Logic (Still FP32)
        new_mag = F.relu(mag + self.b.float())

        # 4. Reconstruct Phase (Safe Division)
        # (z / mag) is the unit vector (phase)
        phase = z_32 / (mag + 1e-6)

        # 5. Result
        out = new_mag * phase

        # 6. Cast back to network dtype (BF16/FP16)
        return out.to(z.dtype)

class ComplexToRealBridge(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.proj = nn.Linear(d_model * 2, d_model)
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x_complex):
        cat = torch.cat([x_complex.real, x_complex.imag], dim=-1)
        return self.norm(self.proj(cat))

# ==========================================
# 4. DYNAMIC RoSE (Mamba-3 Engine)
# ==========================================
class DynamicRoSE(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, max_period=10000.0):
        super().__init__()
        self.embedding_dim = embedding_dim

        # 1. Master Real Embedding (The "Particle")
        self.raw_embedding = nn.Embedding(num_embeddings, embedding_dim)

        # 2. Complex Adapter (The "Wave" Magnitude/Initial Phase)
        self.adapter = nn.Linear(embedding_dim, embedding_dim * 2)

        # 3. Static Frequencies (Positional)
        freqs = torch.exp(torch.arange(0, embedding_dim, dtype=torch.float32) * -(math.log(max_period) / embedding_dim))
        self.register_buffer('freqs', freqs)

        self.rotation_predictor = nn.Linear(embedding_dim, embedding_dim * 2)

    def forward(self, input_ids):
        # A. Raw Particle
        real_base = self.raw_embedding(input_ids)
        B, L, D = real_base.shape

        # B. Complex Wave Content
        complex_params = self.adapter(real_base)
        z_t = torch.complex(complex_params[..., :D], complex_params[..., D:])

        rot_raw = self.rotation_predictor(real_base)
        rot_x, rot_y = rot_raw.chunk(2, dim=-1)

        rot_mag = torch.sqrt(rot_x**2 + rot_y**2 + 1e-6)
        dynamic_rot = torch.complex(rot_x / rot_mag, rot_y / rot_mag)

        # D. Static Positional Rotation
        pos = torch.arange(L, device=input_ids.device).float()
        static_angles = torch.outer(pos, self.freqs) # [L, D]
        static_rot = torch.polar(torch.ones_like(static_angles), static_angles) # [L, D]

        z_final = z_t * static_rot.unsqueeze(0) * dynamic_rot

        return z_final, real_base

# ==========================================
# 5. HYENA FILTER
# ==========================================
class HyenaNeuralFilter(nn.Module):
    def __init__(self, d_model, max_len=1024, hidden_dim=64):
        super().__init__()
        self.d_model = d_model
        freqs = torch.exp(torch.arange(0, hidden_dim, 2, dtype=torch.float32) * -(math.log(10000.0) / hidden_dim))
        self.register_buffer("freqs", freqs)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, d_model * 2)
        )
    def forward(self, L, device):
        t = torch.linspace(0, 1, steps=L, device=device).unsqueeze(-1)
        emb = torch.cat([torch.sin(t * self.freqs), torch.cos(t * self.freqs)], dim=-1)
        out = self.mlp(emb).view(L, self.d_model, 2)
        return torch.complex(out[..., 0], out[..., 1])

# ==========================================
# 6. GATED HARMONIC CONVOLUTION (Lean)
# ==========================================
# @title 🛠️ Fixed PRISM Layer (Precision-Gated)

# @title 🛠️ Fixed PRISM Layer (Type-Safe)

class GatedHarmonicConvolution(nn.Module):
    def __init__(self, d_model, max_len=1024, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.filter_len = max_len
        self.neural_filter = HyenaNeuralFilter(d_model, max_len=max_len)
        self.gate_proj = nn.Linear(d_model * 2, d_model * 2)

        self.mix_real = nn.Linear(d_model, d_model)
        self.mix_imag = nn.Linear(d_model, d_model)
        self.out_real = nn.Linear(d_model, d_model)
        self.out_imag = nn.Linear(d_model, d_model)

        self.activation = ModReLU(d_model)
        self.norm = RobustPhaseNorm(d_model)
        self.dropout = ComplexDropout(dropout)

    def forward(self, x, src_mask=None):
        residual = x
        x_norm = self.norm(x)
        if src_mask is not None:
             x_norm = x_norm.masked_fill(src_mask.unsqueeze(-1), 0.0)

        # 🛑 PRECISION GATE 🛑
        # Force operations to Float32 Complex to preserve Phase Physics
        with torch.amp.autocast('cuda', enabled=False):

            # --- THE FIX IS HERE ---
            # Old: x_32 = x_norm.float()  <-- This stripped the imaginary part
            # New: Explicit cast to Complex64
            x_32 = x_norm.to(torch.complex64)
            # -----------------------

            B, L, D = x_32.shape
            eff_L = min(L, self.filter_len)

            # 1. FFT (Now safe because x_32 is definitely complex)
            x_freq = torch.fft.fft(x_32, n=eff_L, dim=1, norm='ortho')

            # 2. Filter (Ensure filter is also complex64)
            h = self.neural_filter(eff_L, x.device).unsqueeze(0).to(torch.complex64)
            x_filtered = x_freq * h

            # 3. IFFT
            x_time = torch.fft.ifft(x_filtered, n=eff_L, dim=1, norm='ortho')

            if L > eff_L: x_time = F.pad(x_time, (0,0,0,L-eff_L))
            else: x_time = x_time[:, :L, :]

            # 4. Gating (Sigmoid logic)
            # Safe concatenation because x_32 is complex64
            x_cat = torch.cat([x_32.real, x_32.imag], dim=-1)

            # Cast weights to Float32 for the calculation
            gate_w = self.gate_proj.weight.to(torch.float32)
            gate_b = self.gate_proj.bias.to(torch.float32)

            gate_out = F.linear(x_cat, gate_w, gate_b)
            gates = torch.sigmoid(gate_out)

            g_r, g_i = gates.chunk(2, dim=-1)
            x_gated_32 = torch.complex(x_time.real * g_r, x_time.imag * g_i)

            # 🏁 EXIT GATE: Cast back to original dtype (likely BFloat16 from autocast)
            # We cast real/imag separately to be safe
            target_dtype = x.dtype
            # If x was complex, target is complex. If x was real, we have an issue.
            # Assuming x comes from autocast, it might be complex16.

            x_gated = x_gated_32.to(target_dtype)

        # 5. Mixing (Back in mixed precision)
        mr, mi = self.mix_real, self.mix_imag
        x_mixed = torch.complex(mr(x_gated.real) - mi(x_gated.imag), mr(x_gated.imag) + mi(x_gated.real))

        x_act = self.activation(x_mixed)

        or_, oi = self.out_real, self.out_imag
        out = torch.complex(or_(x_act.real) - oi(x_act.imag), or_(x_act.imag) + oi(x_act.real))

        return self.dropout(out) + residual
# ==========================================
# 7. MODEL WRAPPERS
# ==========================================
class PRISMEncoder(nn.Module):
    def __init__(self, num_layers, d_model, max_len, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            GatedHarmonicConvolution(d_model, max_len, dropout)
            for _ in range(num_layers)
        ])
        self.final_norm = RobustPhaseNorm(d_model)
    def forward(self, x, src_mask=None):
        for layer in self.layers:
            if self.training: x = torch.utils.checkpoint.checkpoint(layer, x, src_mask, use_reentrant=False)
            else: x = layer(x, src_mask)
        return self.final_norm(x)

class PRISM_WikiText_Model(nn.Module):
    def __init__(self, vocab_size, d_model, max_len, prism_depth=5, trans_depth=1, dropout=0.1):
        super().__init__()
        self.d_model = d_model

        # 1. PRISM Core (The Optical/Passive Part)
        self.rose = DynamicRoSE(vocab_size, d_model)
        self.prism_encoder = PRISMEncoder(prism_depth, d_model, max_len=max_len, dropout=dropout)
        self.bridge = ComplexToRealBridge(d_model)
        self.periscope_proj = nn.Sequential(nn.Linear(d_model * 2, d_model), nn.LayerNorm(d_model), nn.GELU())

        # 2. Refiner (The Digital/Active Part)
        # 🔄 SWAPPED: Replaced Standard Transformer with RoPE-Enabled Encoder
        if trans_depth > 0:
            self.refiner = Encoder(
                dim=d_model,
                depth=trans_depth,
                heads=8,
                rotary_pos_emb=True,
                attn_flash=True,
                attn_dropout=dropout,
                ff_dropout=dropout,

            )
        else:
            self.refiner = None

        # 3. Output
        self.lm_head = nn.Linear(d_model, vocab_size)
        self.lm_head.weight = self.rose.raw_embedding.weight

    def forward(self, input_ids):
        # A. Wave Physics
        wave_src, particle_src = self.rose(input_ids)
        wave_out = self.prism_encoder(wave_src)
        wave_real = self.bridge(wave_out)

        # B. Interface
        mixed_memory = self.periscope_proj(torch.cat([wave_real, particle_src], dim=-1))

        # C. Digital Refinement (Now with RoPE)
        if self.refiner:
            out = self.refiner(mixed_memory)
        else:
            out = mixed_memory

        return self.lm_head(out)

# ==========================================
# 1. SENSORY STREAM (Transformer + RoPE)
# ==========================================
class SensoryStream(nn.Module):
    def __init__(self, depth, d_model, dropout=0.1):
        super().__init__()
        self.encoder = Encoder(
            dim=d_model,
            depth=depth,
            heads=4,                # 256 dim / 64 head_dim = 4 heads
            attn_flash=True,        # Flash Attention
            rotary_pos_emb=True,    # <--- CRITICAL: RoPE Enabled
            attn_dropout=dropout,
            ff_dropout=dropout,
            use_rmsnorm=True,       # RMSNorm (Llama style)
            ff_glu=True             # SwiGLU (Llama style)
        )

    def forward(self, x):
        return self.encoder(x)

# ==========================================
# 2. PILLARS-DAT (Dual Attention with RoPE)
# ==========================================
class Pillars_DAT(nn.Module):
    def __init__(self, vocab_size, d_model=512, d_branch=256, seq_len=4096, depth=4):
        super().__init__()
        self.d_model = d_model
        self.d_branch = d_branch

        # --- A. SHARED ROOT ---
        self.rose = DynamicRoSE(vocab_size, d_model)

        # --- B. DOWNSAMPLE ---
        self.particle_down = nn.Linear(d_model, d_branch)
        self.wave_down = nn.Linear(d_model * 2, d_branch * 2)

        # --- C. STREAM 1: SENSORY (Object Attributes) ---
        # REPLACED: FNet -> Transformer with RoPE
        # NOTE: No self.sensory_pos anymore! RoPE handles it.
        self.stream_sensory = SensoryStream(depth=depth, d_model=d_branch, dropout=0.1)

        # --- D. STREAM 2: RELATIONAL (Structure / Phase) ---
        # PRISM handles positions internally via RoSE frequencies
        self.stream_relational = PRISMEncoder(num_layers=depth, d_model=d_branch, max_len=seq_len, dropout=0.1)
        self.relational_bridge = ComplexToRealBridge(d_branch)

        # --- E. FUSION ---
        self.fusion_proj = nn.Linear(d_branch * 2, d_model)
        self.fusion_norm = nn.LayerNorm(d_model)

        # --- F. REFINER ---
        self.refiner = Encoder(
            dim=d_model, depth=1, heads=8, attn_flash=True,
            rotary_pos_emb=True, attn_dropout=0.1, ff_dropout=0.1
        )

        # --- G. OUTPUT ---
        self.head_bias = nn.Parameter(torch.zeros(vocab_size))

    def forward(self, input_ids):
        # 1. Root Physics
        wave_src, particle_src = self.rose(input_ids)

        # 2. Downsample
        p_small = self.particle_down(particle_src)

        # Prepare complex wave input
        w_flat = torch.cat([wave_src.real, wave_src.imag], dim=-1)
        w_small_flat = self.wave_down(w_flat)
        w_small = torch.complex(w_small_flat[..., :self.d_branch], w_small_flat[..., self.d_branch:])

        # 3. Parallel Processing

        # --- Stream A: Sensory (Transformer + RoPE) ---
        # Pass pure features. RoPE adds position info inside the attention layer.
        sensory_out = self.stream_sensory(p_small)

        # --- Stream B: Relational (PRISM) ---
        relational_out_complex = self.stream_relational(w_small)
        relational_out = self.relational_bridge(relational_out_complex)

        # 4. Integration
        stacked = torch.cat([sensory_out, relational_out], dim=-1)
        context = self.fusion_norm(self.fusion_proj(stacked))

        # 5. Refinement
        refined = self.refiner(context)

        # 6. Output
        logits = F.linear(refined, self.rose.raw_embedding.weight, self.head_bias)

        return logits

import torch
import torch.nn as nn
from prettytable import PrettyTable # Optional, but makes tables nice.
# If you don't have prettytable, the code below uses standard f-strings.

import torch
import torch.nn as nn

import torch
import torch.nn as nn

def deep_analyze_pillars(model):
    def get_p(obj):
        """Safely returns parameter count for Modules OR raw Parameters."""
        if isinstance(obj, nn.Parameter):
            return obj.numel()
        return sum(p.numel() for p in obj.parameters() if p.requires_grad)

    def format_num(n):
        if n > 1e6: return f"{n/1e6:.2f}M"
        if n > 1e3: return f"{n/1e3:.2f}K"
        return str(n)

    print("\n" + "="*80)
    print(f"🏗️  PILLARS (COMPACT) - DEEP LAYER ANALYSIS")
    print("="*80)
    print(f"{'MODULE / LAYER':<40} | {'PARAMS':<15} | {'TYPE'}")
    print("-" * 80)

    total_params = get_p(model)

    # -----------------------------------------------
    # 1. STATIC MEMORY (Embeddings)
    # -----------------------------------------------
    vocab_emb = get_p(model.rose.raw_embedding)
    fnet_pos  = get_p(model.fnet_pos)

    print(f"{'Shared Vocab Embedding':<40} | {format_num(vocab_emb):<15} | 💾 STORAGE")
    print(f"{'FNet Positional Embedding':<40} | {format_num(fnet_pos):<15} | 💾 STORAGE")

    # -----------------------------------------------
    # 2. INPUT LOGIC (RoSE & Downsampling)
    # -----------------------------------------------
    rose_total = get_p(model.rose)
    rose_logic = rose_total - vocab_emb # Subtract the embedding matrix we already counted

    print("-" * 80)
    print(f"{'Dynamic RoSE (Adapters)':<40} | {format_num(rose_logic):<15} | 🌊 PHASE INIT")
    print(f"{'Particle Downsample (512->384)':<40} | {format_num(get_p(model.particle_down)):<15} | 📉 PROJ")
    print(f"{'Wave Downsample (1024->768)':<40} | {format_num(get_p(model.wave_down)):<15} | 📉 PROJ")

    # -----------------------------------------------
    # 3. STREAM A: RATE (FNet)
    # -----------------------------------------------
    print("-" * 80)
    print(f"TRACK A: RATE STREAM (FNet) - Depth {len(model.stream_rate.layers)}")

    fnet_encoder_total = 0
    for i, layer in enumerate(model.stream_rate.layers):
        p = get_p(layer)
        fnet_encoder_total += p
        print(f"  ├─ FNet Block {i:<24} | {format_num(p):<15} | ⚡ RATE")

    fnet_norm = get_p(model.stream_rate.norm_out)
    fnet_encoder_total += fnet_norm
    print(f"  └─ Final Norm {i:<24} | {format_num(fnet_norm):<15} | ⚡ RATE")

    # -----------------------------------------------
    # 4. STREAM B: PHASE (PRISM)
    # -----------------------------------------------
    print("-" * 80)
    print(f"TRACK B: PHASE STREAM (PRISM) - Depth {len(model.stream_phase.layers)}")

    prism_encoder_total = 0
    for i, layer in enumerate(model.stream_phase.layers):
        p = get_p(layer)
        prism_encoder_total += p
        print(f"  ├─ PRISM Block {i:<23} | {format_num(p):<15} | 🌊 PHASE")

    prism_norm = get_p(model.stream_phase.final_norm)
    prism_encoder_total += prism_norm
    print(f"  └─ Final Norm {i:<24} | {format_num(prism_norm):<15} | 🌊 PHASE")

    bridge_p = get_p(model.phase_bridge)
    print(f"{'Phase Bridge (Complex->Real)':<40} | {format_num(bridge_p):<15} | 🌉 BRIDGE")

    # -----------------------------------------------
    # 5. THE BRAIN (Fusion & Refiner)
    # -----------------------------------------------
    print("-" * 80)
    fusion_p = get_p(model.fusion_proj) + get_p(model.fusion_norm)
    print(f"{'Fusion (Concat -> Proj -> Norm)':<40} | {format_num(fusion_p):<15} | 🧠 FUSION")

    refiner_p = get_p(model.refiner)
    print(f"{'Transformer Refiner (1 Layer)':<40} | {format_num(refiner_p):<15} | 🧠 ATTENTION")

    # [FIX] Handle nn.Parameter directly
    head_bias_p = get_p(model.head_bias)
    print(f"{'Output Head Bias':<40} | {format_num(head_bias_p):<15} | 🎯 OUTPUT")

    # -----------------------------------------------
    # 6. SUMMARY
    # -----------------------------------------------
    print("="*80)

    storage = vocab_emb + fnet_pos + head_bias_p
    active = total_params - storage

    print(f"TOTAL PARAMETERS:      {total_params/1e6:.2f} M")
    print(f"   ├─ 💾 Storage:      {storage/1e6:.2f} M  (Embeddings)")
    print(f"   └─ 🧠 Compute:      {active/1e6:.2f} M  (Logic/Weights)")
    print("-" * 80)
    print(f"STREAM BREAKDOWN:")
    print(f"   ├─ ⚡ Rate Stream:   {fnet_encoder_total/1e6:.2f} M")
    print(f"   └─ 🌊 Phase Stream:  {prism_encoder_total/1e6:.2f} M")
    print("="*80 + "\n")

    return total_params


In [ ]:

# Run the parameter analysis to confirm strict adherence to budget
def deep_analyze_pillars_dat(model):
    def get_p(obj):
        if isinstance(obj, nn.Parameter): return obj.numel()
        return sum(p.numel() for p in obj.parameters() if p.requires_grad)

    def format_num(n):
        if n > 1e6: return f"{n/1e6:.2f}M"
        if n > 1e3: return f"{n/1e3:.2f}K"
        return str(n)

    print("\n" + "="*80)
    print(f"🏛️  PILLARS-DAT (Hybrid Transformer-PRISM) - ANALYSIS")
    print("="*80)
    print(f"{'MODULE / LAYER':<40} | {'PARAMS':<12} | {'TYPE'}")
    print("-" * 80)

    total_params = get_p(model)

    # --- 1. MEMORY ---
    vocab_emb = get_p(model.rose.raw_embedding)
    print(f"{'Shared Vocab Embedding':<40} | {format_num(vocab_emb):<12} | 💾 STORAGE")

    # --- 2. INPUT PHYSICS ---
    rose_logic = get_p(model.rose) - vocab_emb
    print(f"{'Dynamic RoSE (Adapters)':<40} | {format_num(rose_logic):<12} | 🌊 PHYSICS")

    down_p = get_p(model.particle_down) + get_p(model.wave_down)
    print(f"{'Stream Splitters (Downsample)':<40} | {format_num(down_p):<12} | 📉 PROJ")

    # --- 3. STREAM A: SENSORY (TRANSFORMER) ---
    print("-" * 80)
    print(f"STREAM A: SENSORY (Identity/Magnitude)")
    sensory_p = get_p(model.stream_sensory)
    # Attempt to count depth if accessible, else generic
    try:
        depth_s = len(model.stream_sensory.encoder.layers)
        print(f"  ├─ Transformer Encoder (Depth {depth_s})     | {format_num(sensory_p):<12} | ⚡ ATTENTION")
    except:
        print(f"  ├─ Transformer Encoder (Fused)           | {format_num(sensory_p):<12} | ⚡ ATTENTION")

    # --- 4. STREAM B: RELATIONAL (PRISM) ---
    print("-" * 80)
    print(f"STREAM B: RELATIONAL (Structure/Phase)")
    relational_core = get_p(model.stream_relational)
    relational_bridge = get_p(model.relational_bridge)

    try:
        depth_r = len(model.stream_relational.layers)
        print(f"  ├─ PRISM Encoder (Depth {depth_r})           | {format_num(relational_core):<12} | 🌊 SPECTRAL")
    except:
        print(f"  ├─ PRISM Encoder (Fused)                 | {format_num(relational_core):<12} | 🌊 SPECTRAL")

    print(f"  └─ Bridge (Complex->Real)                | {format_num(relational_bridge):<12} | 🌉 PROJ")

    # --- 5. FUSION & OUTPUT ---
    print("-" * 80)
    fusion_p = get_p(model.fusion_proj) + get_p(model.fusion_norm)
    print(f"{'Fusion (Concat -> Proj)':<40} | {format_num(fusion_p):<12} | 🧠 MIX")

    refiner_p = get_p(model.refiner)
    print(f"{'Refiner (1-Layer Transformer)':<40} | {format_num(refiner_p):<12} | 🧠 REASONING")

    bias_p = get_p(model.head_bias)
    print(f"{'Output Head Bias':<40} | {format_num(bias_p):<12} | 🎯 OUT")

    # --- SUMMARY ---
    print("="*80)
    storage = vocab_emb + bias_p
    active = total_params - storage

    print(f"TOTAL PARAMETERS:       {total_params/1e6:.2f} M")
    print(f"   ├─ 💾 Storage:       {storage/1e6:.2f} M  (Embeddings)")
    print(f"   └─ 🧠 Compute:       {active/1e6:.2f} M  (Active Weights)")
    print("-" * 80)
    print(f"RATIO CHECK:")
    print(f"   ⚡ Sensory (Transf): {sensory_p/1e6:.2f} M")
    print(f"   🌊 Relation (PRISM): {(relational_core + relational_bridge)/1e6:.2f} M")
    print("="*80 + "\n")


In [ ]:
# ==========================================
# 4. LOGGING & ANALYSIS UTILITIES
# ==========================================
def deep_analyze_pillars_dat(model):
    def get_p(obj):
        if isinstance(obj, nn.Parameter): return obj.numel()
        return sum(p.numel() for p in obj.parameters() if p.requires_grad)

    def format_num(n):
        if n > 1e6: return f"{n/1e6:.2f}M"
        if n > 1e3: return f"{n/1e3:.2f}K"
        return str(n)

    print("\n" + "="*80)
    print(f"🏛️  PILLARS-DAT (Hybrid Transformer-PRISM) - ANALYSIS")
    print("="*80)
    print(f"{'MODULE / LAYER':<40} | {'PARAMS':<12} | {'TYPE'}")
    print("-" * 80)

    total_params = get_p(model)

    # --- 1. MEMORY ---
    vocab_emb = get_p(model.rose.raw_embedding)
    print(f"{'Shared Vocab Embedding':<40} | {format_num(vocab_emb):<12} | 💾 STORAGE")

    # --- 2. INPUT PHYSICS ---
    rose_logic = get_p(model.rose) - vocab_emb
    print(f"{'Dynamic RoSE (Adapters)':<40} | {format_num(rose_logic):<12} | 🌊 PHYSICS")

    down_p = get_p(model.particle_down) + get_p(model.wave_down)
    print(f"{'Stream Splitters (Downsample)':<40} | {format_num(down_p):<12} | 📉 PROJ")

    # --- 3. STREAM A: SENSORY (TRANSFORMER) ---
    print("-" * 80)
    print(f"STREAM A: SENSORY (Identity/Magnitude)")
    sensory_p = get_p(model.stream_sensory)
    try:
        depth_s = len(model.stream_sensory.encoder.layers)
        print(f"  ├─ Transformer Encoder (Depth {depth_s})     | {format_num(sensory_p):<12} | ⚡ ATTENTION")
    except:
        print(f"  ├─ Transformer Encoder (Fused)           | {format_num(sensory_p):<12} | ⚡ ATTENTION")

    # --- 4. STREAM B: RELATIONAL (PRISM) ---
    print("-" * 80)
    print(f"STREAM B: RELATIONAL (Structure/Phase)")
    relational_core = get_p(model.stream_relational)
    relational_bridge = get_p(model.relational_bridge)

    try:
        depth_r = len(model.stream_relational.layers)
        print(f"  ├─ PRISM Encoder (Depth {depth_r})           | {format_num(relational_core):<12} | 🌊 SPECTRAL")
    except:
        print(f"  ├─ PRISM Encoder (Fused)                 | {format_num(relational_core):<12} | 🌊 SPECTRAL")

    print(f"  └─ Bridge (Complex->Real)                | {format_num(relational_bridge):<12} | 🌉 PROJ")

    # --- 5. FUSION & OUTPUT ---
    print("-" * 80)
    fusion_p = get_p(model.fusion_proj) + get_p(model.fusion_norm)
    print(f"{'Fusion (Concat -> Proj)':<40} | {format_num(fusion_p):<12} | 🧠 MIX")

    refiner_p = get_p(model.refiner)
    print(f"{'Refiner (1-Layer Transformer)':<40} | {format_num(refiner_p):<12} | 🧠 REASONING")

    bias_p = get_p(model.head_bias)
    print(f"{'Output Head Bias':<40} | {format_num(bias_p):<12} | 🎯 OUT")

    # --- SUMMARY ---
    print("="*80)
    storage = vocab_emb + bias_p
    active = total_params - storage

    print(f"TOTAL PARAMETERS:       {total_params/1e6:.2f} M")
    print(f"   ├─ 💾 Storage:       {storage/1e6:.2f} M  (Embeddings)")
    print(f"   └─ 🧠 Compute:       {active/1e6:.2f} M  (Active Weights)")
    print("-" * 80)
    print(f"RATIO CHECK:")
    print(f"   ⚡ Sensory (Transf): {sensory_p/1e6:.2f} M")
    print(f"   🌊 Relation (PRISM): {(relational_core + relational_bridge)/1e6:.2f} M")
    print("="*80 + "\n")

def generate_run_id():
    raw = datetime.now().strftime("%Y%m%d%H%M%S%f")
    return hashlib.md5(raw.encode()).hexdigest()[:8]

def log_environment(save_dir, run_id, config):
    log_path = os.path.join(save_dir, f"env_metadata_{run_id}.txt")
    with open(log_path, "w") as f:
        f.write(f"PRISM EXPERIMENT METADATA | Run ID: {run_id}\n{'='*50}\n")
        for k, v in config.items(): f.write(f"{k}: {v}\n")
    print(f"📝 Environment Snapshot saved to: {log_path}")

def log_metrics(save_dir, run_id, epoch, train_loss, val_loss, ppl):
    log_path = os.path.join(save_dir, f"metrics_log_{run_id}.csv")
    if not os.path.exists(log_path):
        with open(log_path, "w") as f: f.write("Timestamp,Epoch,Train_Loss,Val_Loss,Perplexity\n")
    with open(log_path, "a") as f:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        f.write(f"{ts},{epoch},{train_loss:.6f},{val_loss:.6f},{ppl:.6f}\n")

def save_checkpoint(path, model, optimizer, scheduler, scaler, epoch, best_loss, config):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(), # <--- IMPORTANT FOR AMP
        'best_val_loss': best_loss,
        'config': config
    }, path)

# ==========================================
# 5. A100 TRAINING LOOP (WITH LOGGING)
# ==========================================
# ==========================================
# 4. LOGGING & ANALYSIS UTILITIES
# ==========================================
def deep_analyze_pillars_dat(model):
    def get_p(obj):
        if isinstance(obj, nn.Parameter): return obj.numel()
        return sum(p.numel() for p in obj.parameters() if p.requires_grad)

    def format_num(n):
        if n > 1e6: return f"{n/1e6:.2f}M"
        if n > 1e3: return f"{n/1e3:.2f}K"
        return str(n)

    print("\n" + "="*80)
    print(f"🏛️  PILLARS-DAT (Hybrid Transformer-PRISM) - ANALYSIS")
    print("="*80)
    print(f"{'MODULE / LAYER':<40} | {'PARAMS':<12} | {'TYPE'}")
    print("-" * 80)

    total_params = get_p(model)

    # --- 1. MEMORY ---
    vocab_emb = get_p(model.rose.raw_embedding)
    print(f"{'Shared Vocab Embedding':<40} | {format_num(vocab_emb):<12} | 💾 STORAGE")

    # --- 2. INPUT PHYSICS ---
    rose_logic = get_p(model.rose) - vocab_emb
    print(f"{'Dynamic RoSE (Adapters)':<40} | {format_num(rose_logic):<12} | 🌊 PHYSICS")

    down_p = get_p(model.particle_down) + get_p(model.wave_down)
    print(f"{'Stream Splitters (Downsample)':<40} | {format_num(down_p):<12} | 📉 PROJ")

    # --- 3. STREAM A: SENSORY (TRANSFORMER) ---
    print("-" * 80)
    print(f"STREAM A: SENSORY (Identity/Magnitude)")
    sensory_p = get_p(model.stream_sensory)
    try:
        depth_s = len(model.stream_sensory.encoder.layers)
        print(f"  ├─ Transformer Encoder (Depth {depth_s})     | {format_num(sensory_p):<12} | ⚡ ATTENTION")
    except:
        print(f"  ├─ Transformer Encoder (Fused)           | {format_num(sensory_p):<12} | ⚡ ATTENTION")

    # --- 4. STREAM B: RELATIONAL (PRISM) ---
    print("-" * 80)
    print(f"STREAM B: RELATIONAL (Structure/Phase)")
    relational_core = get_p(model.stream_relational)
    relational_bridge = get_p(model.relational_bridge)

    try:
        depth_r = len(model.stream_relational.layers)
        print(f"  ├─ PRISM Encoder (Depth {depth_r})           | {format_num(relational_core):<12} | 🌊 SPECTRAL")
    except:
        print(f"  ├─ PRISM Encoder (Fused)                 | {format_num(relational_core):<12} | 🌊 SPECTRAL")

    print(f"  └─ Bridge (Complex->Real)                | {format_num(relational_bridge):<12} | 🌉 PROJ")

    # --- 5. FUSION & OUTPUT ---
    print("-" * 80)
    fusion_p = get_p(model.fusion_proj) + get_p(model.fusion_norm)
    print(f"{'Fusion (Concat -> Proj)':<40} | {format_num(fusion_p):<12} | 🧠 MIX")

    refiner_p = get_p(model.refiner)
    print(f"{'Refiner (1-Layer Transformer)':<40} | {format_num(refiner_p):<12} | 🧠 REASONING")

    bias_p = get_p(model.head_bias)
    print(f"{'Output Head Bias':<40} | {format_num(bias_p):<12} | 🎯 OUT")

    # --- SUMMARY ---
    print("="*80)
    storage = vocab_emb + bias_p
    active = total_params - storage

    print(f"TOTAL PARAMETERS:       {total_params/1e6:.2f} M")
    print(f"   ├─ 💾 Storage:       {storage/1e6:.2f} M  (Embeddings)")
    print(f"   └─ 🧠 Compute:       {active/1e6:.2f} M  (Active Weights)")
    print("-" * 80)
    print(f"RATIO CHECK:")
    print(f"   ⚡ Sensory (Transf): {sensory_p/1e6:.2f} M")
    print(f"   🌊 Relation (PRISM): {(relational_core + relational_bridge)/1e6:.2f} M")
    print("="*80 + "\n")

def init_pillars_dat_weights(model):
    print("✨ APPLYING PILLARS-DAT INITIALIZATION PROTOCOL...")
    # 1. SHARED ROOT (RoSE)
    nn.init.normal_(model.rose.raw_embedding.weight, std=model.d_model ** -0.5)
    nn.init.orthogonal_(model.rose.adapter.weight)

    # --- ROSE IDENTITY TRICK ---
    nn.init.normal_(model.rose.rotation_predictor.weight, std=0.01)
    with torch.no_grad():
        model.rose.rotation_predictor.bias[:model.d_model].fill_(1.0) # Real=1
        model.rose.rotation_predictor.bias[model.d_model:].fill_(0.0) # Imag=0

    # 2. DOWNSAMPLERS
    nn.init.orthogonal_(model.particle_down.weight, gain=1.414)
    nn.init.orthogonal_(model.wave_down.weight, gain=1.414)

    # 3. SENSORY STREAM (Transformer + RoPE)
    print("   ├─ Initializing Sensory Stream (Transformer)...")
    for name, p in model.stream_sensory.named_parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
        elif "norm" in name.lower() and p.dim() == 1:
            if "weight" in name: nn.init.ones_(p)
            if "bias" in name:   nn.init.zeros_(p)

    # 4. RELATIONAL STREAM (PRISM)
    print("   ├─ Initializing Relational Stream (PRISM)...")
    for name, m in model.stream_relational.named_modules():
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight, gain=1.0)
            if m.bias is not None: nn.init.zeros_(m.bias)
        if isinstance(m, ModReLU):
            nn.init.constant_(m.b, 0.01)

    # 5. FUSION & REFINER
    nn.init.xavier_uniform_(model.fusion_proj.weight, gain=1.0)
    for p in model.refiner.parameters():
        if p.dim() > 1: nn.init.xavier_uniform_(p)

    # 6. TIED HEAD BIAS
    nn.init.zeros_(model.head_bias)
    print("✅ DAT INITIALIZATION COMPLETE.")

# ==========================================
# 5. A100 TRAINING LOOP (WITH LOGGING)
# ==========================================
def run_a100_training(experiment_name="PILLARS_DAT_A100_Final"):
    from torch.cuda.amp import autocast, GradScaler
    from torch.utils.tensorboard import SummaryWriter

    # --- 1. SETUP DRIVE & LOGGING ---
    from google.colab import drive
    if not os.path.exists('/content/drive'): drive.mount('/content/drive')

    run_id = generate_run_id()
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    SAVE_DIR = os.path.join("/content/drive/My Drive/PRISM_Experiments", f"{experiment_name}_{timestamp}_{run_id}")
    os.makedirs(SAVE_DIR, exist_ok=True)

    writer = SummaryWriter(log_dir=SAVE_DIR)

    # Config for Logs
    config_dump = {
        "run_id": run_id, "batch_size": 6, "accum": 8, "d_model": D_MODEL, "depth": DEPTH, "seq_len": SEQ_LEN
    }
    log_environment(SAVE_DIR, run_id, config_dump)

    # --- 2. MODEL & DATA ---
    SAFE_BATCH_SIZE = BATCH_SIZE
    GRAD_ACCUM = 4
    print(f"\n⚡ A100 DETECTED. CONFIGURING FLASH ATTENTION PIPELINE...")

    lm_datasets, data_collator = prepare_data_from_hub()
    train_loader = DataLoader(lm_datasets["train"], batch_size=SAFE_BATCH_SIZE, shuffle=True, collate_fn=data_collator, num_workers=4, pin_memory=True)
    valid_loader = DataLoader(lm_datasets["validation"], batch_size=SAFE_BATCH_SIZE, collate_fn=data_collator, num_workers=2)

    model = Pillars_DAT(vocab_size=VOCAB_SIZE, d_model=D_MODEL, d_branch=D_BRANCH, seq_len=SEQ_LEN, depth=DEPTH).to(DEVICE)
    init_pillars_dat_weights(model)
    print(model)
    deep_analyze_pillars_dat(model) # <--- Parameter Analysis

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
    warmup_steps = int(total_steps * 0.1)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler() # For AMP

    print(f"\n🚀 IGNITING FUSION DRIVE... Saving to: {SAVE_DIR}")

    global_step = 0
    best_val_loss = float('inf')

    for epoch in range(EPOCHS):
        model.train()
        pbar = tqdm(train_loader, desc=f"Ep {epoch+1}")

        for step, batch in enumerate(pbar):
            x, y = batch['input_ids'].to(DEVICE), batch['labels'].to(DEVICE)

            # ⚡ AMP CONTEXT
            with autocast(dtype=torch.float16):
                logits = model(x).view(-1, VOCAB_SIZE)
                loss = criterion(logits, y.view(-1)) / GRAD_ACCUM

            scaler.scale(loss).backward()

            if (step + 1) % GRAD_ACCUM == 0:
                scaler.unscale_(optimizer)
                # 🛑 CALC GRAD NORM HERE FOR PBAR 🛑
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

                # 📝 STEP LOGGING
                actual_loss = loss.item() * GRAD_ACCUM
                writer.add_scalar('Train/Loss', actual_loss, global_step)
                writer.add_scalar('Train/GradNorm', grad_norm.item(), global_step)
                writer.add_scalar('Train/LR', scheduler.get_last_lr()[0], global_step)

                # ✨ UPDATE PBAR WITH GNORM ✨
                pbar.set_postfix({
                    'loss': f"{actual_loss:.4f}",
                    'gnorm': f"{grad_norm.item():.2f}"
                })

        # --- VALIDATION ---
        model.eval()
        val_loss = 0
        with torch.no_grad(), autocast():
            for batch in valid_loader:
                x, y = batch['input_ids'].to(DEVICE), batch['labels'].to(DEVICE)
                val_loss += criterion(model(x).view(-1, VOCAB_SIZE), y.view(-1)).item()

        avg_val_loss = val_loss / len(valid_loader)
        # Prevent overflow if loss is exploding
        ppl = math.exp(avg_val_loss) if avg_val_loss < 20 else float('inf')

        print(f"✨ Ep {epoch+1} | Val Loss: {avg_val_loss:.4f} | PPL: {ppl:.2f}")

        # 📝 EPOCH LOGGING
        writer.add_scalar('Val/Loss', avg_val_loss, epoch+1)
        writer.add_scalar('Val/PPL', ppl, epoch+1)
        log_metrics(SAVE_DIR, run_id, epoch+1, 0.0, avg_val_loss, ppl)

        # 💾 SAVE CHECKPOINTS (Includes Scaler/Optim/Sched)
        save_checkpoint(os.path.join(SAVE_DIR, "last.pt"), model, optimizer, scheduler, scaler, epoch, best_val_loss, config_dump)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            print(f"   🏆 New Best Model! Saving best.pt...")
            save_checkpoint(os.path.join(SAVE_DIR, "best.pt"), model, optimizer, scheduler, scaler, epoch, best_val_loss, config_dump)

    writer.close()
    return model

if __name__ == "__main__":
    run_a100_training()

In [ ]:
from google.colab import runtime
runtime.unassign()